In [3]:
import os, sys

# プロジェクトルートと irsl_rl を検索パスに追加
PROJ = "/home/irsl/Documents/genesis_choreonoid"   # ホスト側で実行している場合
# PROJ = "/userdir"  # コンテナ内カーネルで実行している場合はこちら
for p in (PROJ, os.path.join(PROJ, "irsl_rl")):
    if p not in sys.path:
        sys.path.insert(0, p)

print("sys.path head:", sys.path[:3])


sys.path head: ['/home/irsl/Documents/genesis_choreonoid/irsl_rl', '/home/irsl/Documents/genesis_choreonoid', '']


In [1]:
import argparse
import os
import pickle

from importlib import metadata
import torch
try:
    try:
        if metadata.version("rsl-rl"):
            raise ImportError
    except metadata.PackageNotFoundError:
        if metadata.version("rsl-rl-lib") != "3.1.1":  #2.2.4
            raise ImportError
except (metadata.PackageNotFoundError, ImportError) as e:
    raise ImportError("Please uninstall 'rsl_rl' and install 'rsl-rl-lib==2.2.4'.") from e
from rsl_rl.runners import OnPolicyRunner

In [5]:
from bp000_env_cnoid import BP000Env as RLEnv

ModuleNotFoundError: No module named 'cnoid'

In [ ]:
# 任意設定項目
exp_name = 'simple-collision-walking'  # ckpt = 2000
ckpt = 1000

action_scale = 0.0 # 動作のスケールを調整

In [ ]:
## set robot path fix collisiton 
ROOT = os.path.abspath(os.path.join(os.path.dirname(__file__), ".."))  # /userdir
robot_path = os.path.join(ROOT, "userdir", "humanoid_research_k", "robots", "kawada_base.simple_collision.urdf")

log_dir = f"logs/{args.exp_name}"
env_cfg, obs_cfg, reward_cfg, command_cfg, train_cfg = pickle.load(open(f"logs/{args.exp_name}/cfgs.pkl", "rb"))
reward_cfg["reward_scales"] = {}

In [ ]:
## override
env_cfg["episode_length_s"] = 40.0
command_cfg["lin_vel_x_range"] = [0.5, 0.5]
env_cfg['dt'] = 0.01
env_cfg['substeps'] = 4

In [ ]:
env = RLEnv(
    num_envs=1,
    env_cfg=env_cfg,
    obs_cfg=obs_cfg,
    reward_cfg=reward_cfg,
    command_cfg=command_cfg,
    dt=env_cfg['dt'],
    substeps=env_cfg['substeps'],
    show_viewer=True,
    robot_urdf_path=robot_path,
)

In [ ]:
runner = OnPolicyRunner(env, train_cfg, log_dir, device='cuda')
resume_path = os.path.join(log_dir, f"model_{args.ckpt}.pt")
runner.load(resume_path)
policy = runner.get_inference_policy(device='cuda')

obs, _ = env.reset()
cnt = 0

In [ ]:
with torch.no_grad():
    print("cnt :", cnt)   
    actions = policy(obs)

    # アクションに倍率を適用して動きを制限
    scaled_actions = actions * action_scale

    print("Original actions : ", actions)
    print("Scaled actions : ", scaled_actions)

    obs, rews, dones, infos = env.step(scaled_actions) # スケール済みアクションを使用
    print("OBS : "obs)
    
    cnt += 1

print(f"データ収集: step {cnt}")